Загрузка и подключение библиотек; настройка random_seed для воспроизводимости результатов; подключение гугл диска

In [1]:
!pip install -q python-docx
!pip install -q transformers datasets accelerate peft trl bitsandbytes bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.5 MB/s eta 0:00:00


In [2]:
import os
import re
import gc
import torch
import pandas as pd
import numpy as np
from docx import Document
from datasets import Dataset
from tqdm import tqdm
import random
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
    PeftModel
)
from bert_score import BERTScorer
import json
import matplotlib.pyplot as plt
import glob

def set_random_seed(seed: int = 27):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True

torch.cuda.empty_cache()
gc.collect()

156

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Конфигурация

In [4]:
set_random_seed(27)

class Config:
    DATA_FILES = [
        ('pul_intervyu_1.docx', 'kodirovki_pfi_2023.docx'),
        ('pul_intervyu_2.docx', 'kodirovki_pfi_2024.docx'),
        ('pul_intervyu_3.docx', 'kodirovki_sp_2024.docx'),
        ('pul_intervyu_4.docx', 'kodirovki_pfi_2025.docx'),
    ]

    MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

    # LoRA параметры
    LORA_R = 16
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.05
    LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

    # Обучение
    BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 4
    LEARNING_RATE = 2e-4
    EPOCHS = 5
    MAX_LENGTH = 768
    MAX_TARGET_LENGTH = 1024

    OUTPUT_DIR = "./qwen_7b_coding"
    LOGGING_STEPS = 10
    SAVE_STEPS = 100
    EVAL_STEPS = 50

Функция для загрузки данных интервью и кодировок (правильных ответов модели)

In [5]:
def parse_interview_transcript(filename):
    """Парсинг транскриптов интервью"""
    try:
        doc = Document(filename)
    except Exception as e:
        print(f"Ошибка при открытии {filename}: {e}")
        return []

    paragraphs = [para.text.strip() for para in doc.paragraphs if para.text.strip()]

    if not paragraphs:
        return []

    global_topic = ""
    if not re.match(r'^Интервью\s+\d+', paragraphs[0]):
        global_topic = paragraphs[0]

    interviews = []
    current_interview = None

    for text in paragraphs:
        if text == global_topic:
            continue

        match = re.match(r'^Интервью\s+(\d+)', text)
        if match:
            if current_interview:
                interviews.append(current_interview)
            interview_num = match.group(1)
            current_interview = {
                'interview_num': interview_num,
                'topic': global_topic,
                'transcript': ''
            }
        elif current_interview:
            if current_interview['transcript']:
                current_interview['transcript'] += '\n' + text
            else:
                current_interview['transcript'] = text

    if current_interview:
        interviews.append(current_interview)

    return interviews


def parse_coding(filename):
    """Парсинг кодировок интервью"""
    try:
        doc = Document(filename)
    except Exception as e:
        print(f"Ошибка при открытии {filename}: {e}")
        return []

    paragraphs = [para.text.strip() for para in doc.paragraphs if para.text.strip()]

    interviews = []
    current_interview = None

    for text in paragraphs:
        match = re.match(r'^Интервью\s+(\d+)', text)
        if match:
            if current_interview:
                interviews.append(current_interview)
            interview_num = match.group(1)
            current_interview = {
                'interview_num': interview_num,
                'coding': ''
            }
        elif current_interview:
            if current_interview['coding']:
                current_interview['coding'] += '\n' + text
            else:
                current_interview['coding'] = text

    if current_interview:
        interviews.append(current_interview)

    return interviews


def load_all_data():
    """Загрузка и объединение всех данных"""
    all_interviews = []

    for trans_file, code_file in Config.DATA_FILES:
        print(f"Обработка {trans_file} и {code_file}...")

        transcripts = parse_interview_transcript(trans_file)
        codings = parse_coding(code_file)

        if not transcripts:
            print(f"Предупреждение: Не удалось загрузить транскрипты из {trans_file}")
            continue

        coding_dict = {c['interview_num']: c['coding'] for c in codings}

        for trans in transcripts:
            interview_num = trans['interview_num']
            coding = coding_dict.get(interview_num, '')

            if coding:
                transcript = trans['transcript']
                if len(transcript) > 3000:
                    transcript = transcript[:3000] + "..."

                all_interviews.append({
                    'id': len(all_interviews),
                    'topic': trans['topic'],
                    'interview_num': interview_num,
                    'transcript': transcript,
                    'coding': coding
                })

    df = pd.DataFrame(all_interviews)
    print(f"Всего загружено интервью с кодировками: {len(df)}")
    return df

Функция для создания промпта для обучения модели

In [6]:
def create_training_prompt(example, prompt_example):
    """Создание промта для обучения с примерами"""

    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Example of an input:
{prompt_example['transcript']}

Example of a topic:
{prompt_example['topic']}

Example of the output (you should follow the format, pay attention to constants 'Общий код' and 'конкретный код'):
{prompt_example['coding']}

Now you should do the markup for the interview. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
Give the answer in Russian, as in an example.
"""

    prompt = f"""<|im_start|>system
{instruction}<|im_end|>
<|im_start|>user

Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку и кодирование интервью в указанном формате.<|im_end|>
<|im_start|>assistant
{example['coding']}<|im_end|>
"""

    return prompt

Подготовка датасета для обучения: выделение "эталонного" интервью, закодированного лабораторией, а также разделение выборки на тренировочную, валидационную и тестовую

In [7]:
prompt_example = None

def prepare_dataset(df):
    """Подготовка датасета для обучения"""
    data = []

    global prompt_example
    prompt_example = df.iloc[0]
    df = df.iloc[1:]

    for _, row in df.iterrows():
        if pd.notna(row['coding']) and row['coding'].strip():
            prompt = create_training_prompt(row, prompt_example)
            data.append({'text': prompt})

    return pd.DataFrame(data)

def split_dataset(df):
    """Разделение данных на train/val/test"""
    np.random.seed(42)
    indices = np.random.permutation(len(df))

    n_train = int(0.7 * len(df))
    n_val = int(0.15 * len(df))

    train_df = df.iloc[indices[:n_train]]
    val_df = df.iloc[indices[n_train:n_train + n_val]]
    test_df = df.iloc[indices[n_train + n_val:]]

    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    return train_df, val_df, test_df

Функция для настройки токенизатора

In [8]:
def setup_tokenizer(model_name):
    """Настройка токенизатора"""
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        padding_side="right"
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer

Функция для настройки параметров модели (в частности, ее загрузки) и параметров дообучения

In [9]:
def setup_model_and_lora(model_name, tokenizer):
    """Настройка модели - 4-bit квантизация + LoRA"""

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_storage=torch.uint8,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16
    )

    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    # LoRA конфигурация
    lora_config = LoraConfig(
        r=Config.LORA_R,
        lora_alpha=Config.LORA_ALPHA,
        target_modules=Config.LORA_TARGET_MODULES,
        lora_dropout=Config.LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    model = get_peft_model(model, lora_config)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}% of {total_params:,})")

    return model

Функция для оценки модели на тестовых данных

In [10]:
def evaluate_model(model, tokenizer, test_df, device, num_samples=5):
    """Оценка модели на тестовых данных"""
    model.eval()

    results = []

    for idx, row in test_df.head(num_samples).iterrows():
        text = row['text']
        parts = text.split('<|im_start|>assistant\n')

        if len(parts) > 1:
            prompt = parts[0] + '<|im_start|>assistant\n'
            target = parts[1].replace('<|im_end|>', '').strip()
        else:
            continue

        if len(prompt) > 1500:
            prompt = prompt[:1500] + '<|im_start|>assistant\n'

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                          max_length=1500).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=Config.MAX_TARGET_LENGTH,
                num_beams=5,
                early_stopping=True,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                                      skip_special_tokens=True)

        results.append({
            'id': row.get('id', idx),
            'generated': generated[:800],
            'target': target[:500]
        })

    return results

def generate_coding(model, tokenizer, topic, transcript, prompt_example, device=None):
    """Генерация кодировки для нового интервью"""
    if device is None:
        device = next(model.parameters()).device

    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Example of an input:
{prompt_example['transcript']}

Example of a topic:
{prompt_example['topic']}

Example of the output (you should follow the format, pay attention to constants 'Общий код' and 'конкретный код'):
{prompt_example['coding']}

Now you should do the markup for the interview. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
Give the answer in Russian, as in an example.
"""

    prompt = f"""<|im_start|>system
{instruction}<|im_end|>
<|im_start|>user

Тема интервью:
{topic}

Текст интервью:
{transcript}

Пожалуйста, выполни разметку и кодирование интервью в указанном формате.<|im_end|>
<|im_start|>assistant
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                      max_length=1800).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=Config.MAX_TARGET_LENGTH,
            num_beams=5,
            early_stopping=True,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:],
                                  skip_special_tokens=True)

    return generated

Функция для обучения модели

In [11]:
def main():
    df_interviews = load_all_data()

    if len(df_interviews) == 0:
        return None, None, None

    df_data = prepare_dataset(df_interviews)
    print(f"Создано {len(df_data)} примеров")

    train_df, val_df, test_df = split_dataset(df_data)
    test_df.to_csv('test_data.csv', index=False)

    tokenizer = setup_tokenizer(Config.MODEL_NAME)

    model = setup_model_and_lora(Config.MODEL_NAME, tokenizer)

    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            max_length=Config.MAX_LENGTH,
            padding="max_length"
        )

    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
    val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir=Config.OUTPUT_DIR,
        num_train_epochs=Config.EPOCHS,
        per_device_train_batch_size=Config.BATCH_SIZE,
        per_device_eval_batch_size=Config.BATCH_SIZE,
        gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,
        warmup_ratio=0.05,
        learning_rate=Config.LEARNING_RATE,
        fp16=True,
        logging_steps=Config.LOGGING_STEPS,
        eval_strategy="steps",
        eval_steps=Config.EVAL_STEPS,
        save_steps=Config.SAVE_STEPS,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        max_grad_norm=0.3,
        weight_decay=0.01,
        dataloader_num_workers=0,
    )

    effective_batch_size = Config.BATCH_SIZE * Config.GRADIENT_ACCUMULATION_STEPS

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
    )


    trainer.train()
    trainer.save_model(Config.OUTPUT_DIR)
    tokenizer.save_pretrained(Config.OUTPUT_DIR)

    device = next(model.parameters()).device
    eval_results = evaluate_model(model, tokenizer, test_df, device, num_samples=5)

    if eval_results:
        results_df = pd.DataFrame(eval_results)
        results_df.to_csv('test_evaluation_results.csv', index=False)

        print("\n Пример генерации:")
        example = eval_results[0]
        print(f"Сгенерированный ответ:\n{example['generated'][:800]}...")

    print("Обучение завершено")
    print(f"Модель сохранена в {Config.OUTPUT_DIR}")
    return model, tokenizer, trainer

Обучение модели

In [12]:
set_random_seed(27)

if __name__ == "__main__":
    model, tokenizer, trainer = main()

    if model:

        print("Тестовый запуск:")
        test_topic = "ПОКОЛЕНИЕ Z В ПОИСКАХ БАЛАНСА"
        test_transcript = """
        Интервьюер: Расскажи о своей работе.
        Информант: Мне очень нравится моя работа. У нас отличный коллектив,
        всегда можно обратиться за помощью. И есть возможность развиваться
        профессионально, компания оплачивает курсы.
        """
        result = generate_coding(model, tokenizer, test_topic, test_transcript, prompt_example)
        print(f"\nРезультат:\n{result}")

Обработка pul_intervyu_1.docx и kodirovki_pfi_2023.docx...
Обработка pul_intervyu_2.docx и kodirovki_pfi_2024.docx...
Обработка pul_intervyu_3.docx и kodirovki_sp_2024.docx...
Обработка pul_intervyu_4.docx и kodirovki_pfi_2025.docx...
Всего загружено интервью с кодировками: 150
Создано 149 примеров
Train: 104, Val: 22, Test: 23


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/22 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
50,0.000096,0.000076
100,0.000041,0.000040


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



 Пример генерации:
Сгенерированный ответ:
Интервьюер: А они какого возраста?
Информант: Они гораздо старше меня. Сестре тридцать, если не соврать, тридцать шесть. Брату за 40 уже. Вот. То есть, ну, он на 18 лет меня старше. Вот. Ну вот они как-то вместе с бабушкой меня воспитывали, пока мама работала. Вот. Бабушка сейчас умерла, но вот как-то между собой мы вот общаемся как семья. Вот. Какие-то основные данные.
Интервьюер: А бабушка сейчас здесь?
Информант: Нет, бабушка умерла. Вот как раз завтра будет годовщина, четыре года. Вот. Ну, как бы ей уже много лет было просто. Вот.
Интервьюер: Поняла. Давай тогда еще немножко вернемся в прошлое. Можешь рассказать, во что ты росла, чем занималась, какую семью у тебя было, где ты жила?
Информант: Ну, мы в общем-то все отсюда. У нас семья переехала с Украины. Вот. А мы, там мама моя она уж...
Обучение завершено
Модель сохранена в ./qwen_7b_coding
Тестовый запуск:

Результат:
 обстоятельствами (конкретный код)**
"Ну, мы в общем-то все отсюда. У

Видно, что модель зацикливается в ответах, повторяет цитаты. Исправим это с помощью новой функции генерации, которая штрафует за повторения

In [13]:
from peft import PeftModel

In [15]:
set_random_seed(27)

def clear_cuda_cache():
    """Очистка кэша CUDA памяти"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        return allocated, reserved
    return 0, 0

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_PATH = "./qwen_7b_coding"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

clear_cuda_cache()


model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, is_trainable=False)
model.eval()
clear_cuda_cache()


def generate_without_repetition(prompt, max_new_tokens=512):
    """Генерация с очисткой памяти и удалением повторов"""
    clear_cuda_cache()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=5,
            early_stopping=True,
            do_sample=False,
            repetition_penalty=1.5,
            no_repeat_ngram_size=5,
            penalty_alpha=0.6,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    lines = generated.strip().split('\n')
    seen = set()
    unique_lines = []

    for line in lines:
        if not line.strip():
            continue
        key = line[:50]
        if key not in seen:
            seen.add(key)
            unique_lines.append(line)
        else:
            break

    clear_cuda_cache()
    return '\n'.join(unique_lines[:15])


test_df = pd.read_csv('test_data.csv')
print(f"Тестовых примеров: {len(test_df)}")

for idx in range(min(2, len(test_df))):
    print(f"\n Пример {idx + 1}")
    full_text = test_df.iloc[idx]['text']
    parts = full_text.split('<|im_start|>assistant\n')
    prompt = parts[0] + '<|im_start|>assistant\n'

    print("\n Предсказание модели:")
    generated = generate_without_repetition(prompt)
    print(generated)
    clear_cuda_cache()

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

Обучение не смогло пройти на GPU в Google Colab, так как для данной модели не хватило памяти.

Сохранение модели на Диск

In [16]:
import os
import json

def save_model_complete(model, tokenizer, output_dir="./qwen_coding_7b_model"):
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)

    tokenizer.save_pretrained(output_dir)

    config = {
        "base_model": Config.MODEL_NAME,
        "lora_r": Config.LORA_R,
        "lora_alpha": Config.LORA_ALPHA,
        "max_length": Config.MAX_LENGTH,
        "model_type": "qwen_lora"
    }

    with open(f"{output_dir}/model_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)

    print(f"Модель сохранена в {output_dir}")
    return output_dir

save_model_complete(model, tokenizer, "/content/drive/MyDrive/qwen_coding_7b_model")

# Для загрузки в другом ноутбуке
#model, tokenizer = load_saved_model("/content/drive/MyDrive/qwen_coding_7b_model")

Модель сохранена в /content/drive/MyDrive/qwen_coding_7b_model


'/content/drive/MyDrive/qwen_coding_7b_model'